# Create Princess of Asturias Awards (PRIZE PATTERN)

Princess/Prince of Asturias Award laureates from the official Fundación Princesa de Asturias site.

**Prerequisite:** run `scripts/local/princess_asturias_to_s3.py` to fetch the official full laureate list, fetch the linked detail pages, verify the current award-composition FAQ, and upload parquet to S3.

**Source:** `https://www.fpa.es/en/cargarAplicacionPremiadoCompleto.do` plus linked official laureate detail pages. The script also verifies `https://www.fpa.es/en/area-of-communication-and-media/faqs/princess-of-asturias-awards/` for the current cash-prize rule.

**S3 location:** `s3://openalex-ingest/awards/princess_asturias/princess_asturias_laureates.parquet`

**Awarding body:** Fundación Princesa de Asturias - `F4320323780`

Funder details from OpenAlex Step 0:

- funder_id: `4320323780`
- display_name: `Fundación Princesa de Asturias`
- ror_id: NULL in OpenAlex API result
- doi: `10.13039/501100006336`

Prize-pattern notes:

- `lead_investigator` is the laureate. Institutional/group laureates are preserved as institutional laureates and are not expanded to their members.
- The official site publishes one page per award/year/category. The source script emits one row per `(prize x laureate)` using documented split/no-split rules for shared awards.
- `funder_scheme` is the official prize title, e.g. `Princess of Asturias Award for Technical & Scientific Research`.
- The current FAQ says each award includes EUR 50,000 divided among laureates when shared. Historical per-year/per-laureate cash values are not exposed on the official laureate pages and older Prince-era awards may differ, so this notebook maps `amount = NULL` and keeps `currency = 'EUR'` with an explicit prize-pattern amount waiver.
- `funder_award_id` is a synthetic key built from award year, category, detail page slug, winner index, and laureate slug. The source script hard-fails on duplicate keys.

**Contractor handoff:** local validation was completed without AWS/S3 or Databricks credentials. Admin must upload the parquet, run this notebook, run QA, and then run `CreateAwards.ipynb`. No job YAML is included until admin verification.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.princess_asturias_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/princess_asturias/princess_asturias_laureates.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.princess_asturias_raw;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below. The source script writes all raw columns as strings with `df.astype("string")` before parquet.


In [ ]:
%sql
DESCRIBE openalex.awards.princess_asturias_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.princess_asturias_raw
LIMIT 10;


In [ ]:
%sql
-- Required money-field scan. `current_award_amount_eur` preserves the current FAQ rule,
-- but `source_award_amount` is intentionally NULL because historical values are not published.
SELECT column_name
FROM (DESCRIBE openalex.awards.princess_asturias_raw)
WHERE LOWER(column_name) RLIKE
    'amount|currency|fund|award|value|money|prize|eur|euro|cash|portion';


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(current_award_amount_eur) AS rows_with_current_faq_amount,
    COUNT(source_award_amount) AS rows_with_mapped_source_amount,
    collect_set(currency) AS currencies,
    collect_set(amount_note) AS amount_notes,
    collect_set(faq_url) AS faq_urls
FROM openalex.awards.princess_asturias_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(laureate_name) AS has_laureate_name,
    COUNT(award_year) AS has_award_year,
    COUNT(award_category) AS has_award_category,
    COUNT(prize_title) AS has_prize_title,
    COUNT(citation) AS has_citation,
    COUNT(landing_page_url) AS has_landing_page_url,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.princess_asturias_raw;


In [ ]:
%sql
SELECT
    award_category,
    COUNT(*) AS rows,
    COUNT(DISTINCT landing_page_url) AS official_award_items,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year,
    COUNT(citation) AS rows_with_citation
FROM openalex.awards.princess_asturias_raw
GROUP BY award_category
ORDER BY award_category;


In [ ]:
%sql
SELECT
    winner_count,
    COUNT(DISTINCT landing_page_url) AS official_award_items,
    COUNT(*) AS emitted_laureate_rows
FROM openalex.awards.princess_asturias_raw
GROUP BY winner_count
ORDER BY TRY_CAST(winner_count AS INT);


In [ ]:
%sql
SELECT
    award_year,
    award_category,
    official_laureate_text,
    winner_index,
    winner_count,
    laureate_name,
    laureate_is_organization,
    landing_page_url
FROM openalex.awards.princess_asturias_raw
WHERE TRY_CAST(winner_count AS INT) > 1
ORDER BY TRY_CAST(award_year AS INT) DESC, award_category, landing_page_url, TRY_CAST(winner_index AS INT)
LIMIT 100;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the `CROSS JOIN` would emit zero awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320323780;


## Step 2: Transform to Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.princess_asturias_awards
USING delta
AS
WITH princess_asturias_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323780
), normalized AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS award_year_int,
        NULLIF(TRIM(prize_title), '') AS prize_title_norm,
        NULLIF(TRIM(citation), '') AS citation_norm,
        NULLIF(TRIM(laureate_given_name), '') AS given_name_norm,
        NULLIF(TRIM(laureate_family_name), '') AS family_name_norm,
        NULLIF(TRIM(laureate_name), '') AS laureate_name_norm,
        NULLIF(TRIM(currency), '') AS currency_norm,
        NULLIF(TRIM(landing_page_url), '') AS landing_page_url_norm
    FROM openalex.awards.princess_asturias_raw
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':princess-asturias:', LOWER(n.funder_award_id)))) % 9000000000 AS id,
    CONCAT(CAST(n.award_year_int AS STRING), ' ', n.prize_title_norm, ' - ', n.laureate_name_norm) AS display_name,
    n.citation_norm AS description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount, -- prize-pattern waiver: historical per-laureate amount not exposed by source
    n.currency_norm AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    n.prize_title_norm AS funder_scheme,
    'princess_asturias' AS provenance,
    TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
    n.award_year_int AS start_year,
    n.award_year_int AS end_year,
    struct(
        n.given_name_norm AS given_name,
        n.family_name_norm AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            CAST(NULL AS STRING) AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url_norm AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':princess-asturias:', LOWER(n.funder_award_id)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM normalized n
CROSS JOIN princess_asturias_funder f
WHERE n.funder_award_id IS NOT NULL
  AND n.award_year_int IS NOT NULL
  AND n.prize_title_norm IS NOT NULL
  AND n.laureate_name_norm IS NOT NULL;


In [ ]:
%sql
SELECT COUNT(*) AS transformed_rows
FROM openalex.awards.princess_asturias_awards;


In [ ]:
%sql
-- Duplicate checks. Both values should be zero.
SELECT
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.princess_asturias_awards;


In [ ]:
%sql
-- §6.7 amount/currency waiver check. Prize pattern allows NULL amount when explicitly documented.
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    collect_set(currency) AS currencies,
    COUNT(CASE WHEN currency = 'EUR' THEN 1 END) AS rows_with_currency_eur
FROM openalex.awards.princess_asturias_awards;


In [ ]:
%sql
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year,
    COUNT(description) AS rows_with_description
FROM openalex.awards.princess_asturias_awards
GROUP BY funder_scheme
ORDER BY funder_scheme;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_award_id,
    amount,
    currency,
    funding_type,
    funder_scheme,
    start_year,
    lead_investigator,
    landing_page_url
FROM openalex.awards.princess_asturias_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 25;


## Step 3: Insert into `openalex_awards_raw` at Priority 77


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'princess_asturias' AND priority = 77;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       77 AS priority
FROM openalex.awards.princess_asturias_awards;


## Step 4: QA Inserted Rows


In [ ]:
%sql
SELECT provenance, priority, COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'princess_asturias' AND priority = 77
GROUP BY provenance, priority;


In [ ]:
%sql
SELECT
    COUNT(*) AS inserted_rows,
    COUNT(DISTINCT id) AS distinct_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(description) AS rows_with_description,
    COUNT(amount) AS rows_with_amount,
    collect_set(currency) AS currencies,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'princess_asturias' AND priority = 77;


In [ ]:
%sql
SELECT display_name, funder, funding_type, funder_scheme, start_year, amount, currency, lead_investigator, landing_page_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'princess_asturias' AND priority = 77
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 25;


## Step 5: Run Combined Awards Refresh

After this notebook is verified, run `CreateAwards.ipynb`. That notebook includes the priority-77 registry entry for `princess_asturias` so these rows participate in the global deduped awards table.
